# Cosmos Reason 2 - End-to-end VLM Compression

Deploying a reasoning Vision-Language Model in production is not just a matter of accuracy — it is a matter of cost, latency, and throughput. [Cosmos Reason 2](https://github.com/nvidia-cosmos/cosmos-reason2) is purpose-built for Physical AI: it reasons over images and video to support robotic manipulation, autonomous navigation, and embodied decision-making. These applications demand real-time or near-real-time responses, often on constrained hardware budgets. A 2 B parameter BF16 model is already compact by frontier standards, but even modest compression — 15 % fewer parameters and FP8 weights — can double serving throughput and halve memory footprint, making the difference between a model that fits on a single GPU and one that requires two. The challenge is doing this without sacrificing the chain-of-thought reasoning quality that makes Cosmos Reason 2 valuable in the first place. This notebook shows that with the right pipeline — structured pruning, knowledge distillation, and PTQ — you can reach that bar.

In this course you will learn how to apply the full compression pipeline — prune → distill → quantize → serve → evaluate — on NVIDIA's [`nvidia/Cosmos-Reason2-2B`](https://huggingface.co/nvidia/Cosmos-Reason2-2B), a reasoning-specialised Vision-Language Model (VLM).

**Pruning** focuses on removing parts of the model that are most redundant, **distillation** aims to recover the quality of the pruned model to match the original and **quantization** reduces the precision of weights, activations and KV cache. In this notebook, you will learn how to apply each technique to Cosmos Reason 2 while preserving the accuracy of the model.

**Compression target.** The compressed checkpoint at the end (pruned LM 1.4 B + distilled + FP8 — full VLM ~1.7 B) should land within a couple of points of the BF16 baseline on MathVista and on Cosmos's own Physical AI Bench domain, while not regressing on a general benchmark (RealWorldQA). §6 sets up a 4-way comparison so you can read off `% recovered` for each benchmark. The 15 % cut is on the LM tower (1.7 B → 1.4 B); the vision encoder + projector (0.3 B) is untouched, so the full VLM goes 2.0 B → ~1.7 B. The cut is intentionally conservative — the goal here is *full* recovery.

Pipeline overview:

| Stage | What it does | Wall-time (approx.) |
|---|---|---|
| §1 Download | Pull the gated HF checkpoint (Cosmos-Reason2-2B, ~5 GB; LM tower 1.7 B + vision 0.3 B) | 5 min |
| §2 Prune (NAS) | Minitron NAS on the LM tower, target 1.4 B (15 % cut) | 5–10 min |
| §3 Distill | Text-only KD on the pruned LM tower, ~5 k iters | 20–30 min |
| §4 Quantize | FP8 PTQ with image-aware calibration | 10-15 min |
| §5 Serve | vLLM OpenAI-compatible endpoint | <2 min to spin up |
| §6 Evaluate | 4-way comparison × MathVista / PAI-Bench / RealWorldQA | 20–30 min |



---
## 1.1 Download Cosmos Reason 2 and Inspect the Architecture

> 🔐 Cosmos Reason 2 is **gated** on HuggingFace. Before running §1 request access at the model page and authenticate: `hf auth login --token <YOUR_HF_TOKEN>`.

Pull the checkpoint locally and read its `config.json`. Then we inspect the architecture of the model.

In [ ]:
# Put your HF Token, otherwise this code will throw an error
!hf auth login --token <HF_Token>

BASE_PATH = "/workspace/user_homes/lmikaelyan"

COSMOS_PATH = f"{BASE_PATH}/Cosmos-Reason2-2B"

!hf download nvidia/Cosmos-Reason2-2B --local-dir {COSMOS_PATH}
!ls {COSMOS_PATH} | head

We start by inspecting the architecture of [Cosmos Reason 2](https://github.com/nvidia-cosmos/cosmos-reason2). **Cosmos Reason 2** is a reasoning-specialised VLM built on the **Qwen3-VL** family (`Qwen3VLForConditionalGeneration`). It has three main components:

```
Cosmos-Reason2-2B  (2.44 B total)
├── Vision Encoder   (ViT-style, ~0.42 B)   — frozen during pruning/distill
├── Vision Projector (MLP, ~0.30 B)          — frozen during pruning/distill
└── Language Model   (Qwen3, 1.72 B)         ← compression target
```

A standard VLM architecture looks like the following:

![VLM architecture](VLM.png)

**Language Model tower** (`model.language_model`, 1.72 B):

| Hyperparameter | Value |
| --- | --- |
| `num_hidden_layers` | 28 |
| `hidden_size` | 2048 |
| `num_attention_heads` | 16 |
| `num_key_value_heads` | 8 |
| `head_dim` | 128 |
| `intermediate_size` | 6144 |
| `vocab_size` | 151 936 |
| `max_position_embeddings` | 262 144 |

**Vision Encoder** (24-layer ViT):

| Hyperparameter | Value |
| --- | --- |
| `depth` | 24 |
| `hidden_size` | 1024 |
| `intermediate_size` | 4096 |
| `num_heads` | 16 |
| `patch_size` | 16 × 16 px |
| `spatial_merge_size` | 2 |
| `temporal_patch_size` | 2 |
| `out_hidden_size` | 2048 |
| `deepstack_visual_indexes` | [5, 11, 17] |

**Input resolution** (dynamic):
- `min_pixels`: 65 536 (≈ 256 × 256)
- `max_pixels`: 16 777 216 (≈ 4096 × 4096)
- Images are resized so total pixel count falls in `[min_pixels, max_pixels]` while preserving aspect ratio. After the 2×2 spatial merge, each group of 4 patches maps to one visual token.

Note that we are targeting compression primarily of the LLM component given the size compared to the total size of the model, we will try to decrease the size of it. Research also shows that the vision encoders tend to be very sensitive to compression, so we will omit it here. We aim to decrease the size of the model by 15% in terms of number of parameters. 

Run the code cell below to load the full VLM, resolve the LM tower path (`model.language_model`), count parameters for each component, and print the config values above.

In [ ]:
from transformers import AutoConfig, AutoModelForImageTextToText, AutoProcessor
from modelopt.torch.utils.plugins.vlm import resolve_vlm_lm_path
from IPython.display import display
from PIL import Image
import requests, torch

cfg = AutoConfig.from_pretrained(COSMOS_PATH, trust_remote_code=True)
tc  = cfg.text_config

vlm = AutoModelForImageTextToText.from_pretrained(
    COSMOS_PATH, dtype=torch.bfloat16, device_map="auto", trust_remote_code=True
)
processor = AutoProcessor.from_pretrained(COSMOS_PATH, trust_remote_code=True)

# resolve_vlm_lm_path returns a dotted path like "model.language_model" — the same one
# the prune/distill scripts use to extract / reinsert the LM tower.
lm_path = resolve_vlm_lm_path(vlm)
lm = vlm
for part in lm_path.split("."):
    lm = getattr(lm, part)

total_b = sum(p.numel() for p in vlm.parameters()) / 1e9
lm_b    = sum(p.numel() for p in lm.parameters()) / 1e9
print(f"Total VLM params:               {total_b:.2f} B")
print(f"  Language tower ({lm_path}):  {lm_b:.2f} B")
print(f"  Vision + projector + heads:   {total_b - lm_b:.2f} B")
print()
print(f"architectures:           {cfg.architectures}")
print(f"text_config.model_type:  {tc.model_type}")
print(f"num_hidden_layers:       {tc.num_hidden_layers}")
print(f"hidden_size:             {tc.hidden_size}")
print(f"num_attention_heads:     {tc.num_attention_heads}")
print(f"num_key_value_heads:     {tc.num_key_value_heads}")
print(f"intermediate_size:       {tc.intermediate_size}")

Now let's run a quick sanity-check inference with the **baseline BF16 model** before touching anything. We feed it an [AI2D diagram](https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/transformers/tasks/ai2d-demo.jpg) and ask for a one-sentence caption.

We'll rerun this exact prompt at every compression stage (post-prune, post-distill, post-FP8) so you can track caption drift by eye. Full quantitative evaluation — MathVista, BLINK, RealWorldQA — is deferred to §5.6 once all stages are complete.

> **Expected output**: `"A detailed diagram of a volcano, showcasing its internal structure and external features, including magma movement, volcanic activity, and surrounding geological layers."`
> A coherent, on-topic sentence means the model loaded correctly. Garbled or off-topic output is a red flag — re-check the download in §1.1 before proceeding.

In [ ]:
url = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/transformers/tasks/ai2d-demo.jpg"
image = Image.open(requests.get(url, stream=True).raw).convert("RGB")
preview = image.copy(); preview.thumbnail((512, 512)); display(preview)

messages = [{"role": "user", "content": [
    {"type": "image", "image": image},
    {"type": "text", "text": "Describe the image in one short sentence."},
]}]
inputs = processor.apply_chat_template(
    messages, add_generation_prompt=True, tokenize=True, return_dict=True, return_tensors="pt"
).to(vlm.device)
out = vlm.generate(**inputs, max_new_tokens=64, do_sample=False)
print("Baseline caption:", processor.batch_decode(out[:, inputs['input_ids'].shape[1]:], skip_special_tokens=True)[0])

In [ ]:
# Empty the memory to proceed
import gc; del vlm; gc.collect(); torch.cuda.empty_cache()

---
## 1.2 Prune the LLM

In this section we are going to prune the LLM component of Cosmos Reason 2.

For pruning we are going to apply [`prune_minitron.py`](../prune_minitron.py) on the VLM checkpoint. The script auto-detects the VLM, extracts its language tower to a temporary causal-LM dir, runs activation-importance calibration on `wikitext-2-raw-v1`, prunes the inner LM, then reinserts the pruned LM back into the original VLM container at `--output_hf_path`.

`--prune_export_config` selects the target shape directly. For simplicity we are going to apply **depth pruning** by removing 4 layers from the 28-layer model. You can further apply **width pruning** for shrinking the FFN sizes. 

Megatron-Core has no VLM provider, so [`prune_minitron.py`](../prune_minitron.py) handles this by **extracting** the language tower into a plain HF causal LM, running the regular `mcore_minitron` algorithm on it, then **reinserting** the pruned LM back into the original VLM container. The vision encoder, projector and `lm_head` are copied verbatim.

For more details on model pruning techniques and best practices, check out the [Minitron blog](https://developer.nvidia.com/blog/how-to-prune-and-distill-llama-3-1-8b-to-an-nvidia-llama-3-1-minitron-4b-model/) and accompanying [technical report](https://arxiv.org/pdf/2408.11796). Further pruning recipes and best practices can also be found in the [pruning guidelines notes](https://github.com/NVIDIA/Model-Optimizer/tree/main/examples/pruning#pruning-guidelines).

![Minitron pruning pipeline](Pruning.png)

Expect the run to take roughly 5–10 minutes on a single H100.

> ⚠️ `--hparams_to_skip hidden_size` is **mandatory** here. Without it NAS will happily pick a smaller `hidden_size` to hit the budget, and the reinsertion step will refuse to splice the smaller LM back into the original vision projector (`ValueError` in `reinsert_pruned_lm_into_vlm`), meaning that the projector will need to be retrained. If you intentionally want a smaller hidden_size, drop the flag and plan on retraining the projector separately.


For more advanced pruning, you can apply Neural Architecture Search (NAS). This way we only specify the target size of the model, and the algorithm will then generate candidates with different pruning configs that satisfy this requirement. The candidates then will be evaluated and the best checkpoint will be saved. If you are interested in this type of pruning, see the NAS pruning example in notebook 02. 

In [ ]:
PRUNED_PATH = f"{BASE_PATH}/Cosmos-Reason2-pruned-depth-24"
COSMOS_PATH = f"{BASE_PATH}/Cosmos-Reason2-2B"

!torchrun --nproc_per_node 1 ../prune_minitron.py \
    --pp_size 1 \
    --hf_model_name_or_path {COSMOS_PATH} \
    --prune_export_config '{{"num_layers": 24}}' \
    --calib_dataset_name wikitext \
    --calib_num_samples 2 \
    --output_hf_path {PRUNED_PATH}

Now let's verify that the pruned model matches our target shape and regenerate the caption.

The code cell checks three things:
1. **Parameter count** — the LM tower should be near the `--prune_target_params` budget (~1.4–1.5 B); the vision encoder + projector should be unchanged.
2. **Shape** — We have only updated the `num_hidden_layers`.
3. **Caption** — the pruned model runs the same AI2D prompt as the baseline. Compare the two outputs.

**What to expect.** Don't worry if the caption has changed! Pruning removes layers and shrinks attention/FFN width without any recovery training, so caption quality typically drops — shorter sentences, less detail, occasionally factual drift. This degradation is intentional and expected: the next stage (§5.3 knowledge distillation) is specifically designed to recover it by training the pruned student against the unpruned teacher.

In [ ]:
from transformers import AutoModelForImageTextToText, AutoProcessor
from modelopt.torch.utils.plugins.vlm import resolve_vlm_lm_path
from IPython.display import display
from PIL import Image
import requests, torch

pruned = AutoModelForImageTextToText.from_pretrained(
    PRUNED_PATH, dtype=torch.bfloat16, device_map="auto", trust_remote_code=True
)
processor = AutoProcessor.from_pretrained(PRUNED_PATH, trust_remote_code=True)
tc = pruned.config.text_config

lm_path = resolve_vlm_lm_path(pruned)
lm = pruned
for part in lm_path.split("."):
    lm = getattr(lm, part)
total_b = sum(p.numel() for p in pruned.parameters()) / 1e9
lm_b    = sum(p.numel() for p in lm.parameters()) / 1e9

print(f"Total VLM params:              {total_b:.2f} B")
print(f"  Language tower ({lm_path}): {lm_b:.2f} B  (pruned)")
print(f"  Vision + projector + heads:  {total_b - lm_b:.2f} B  (unchanged)")
print()
print(f"num_hidden_layers:   {tc.num_hidden_layers} (pruned) ")
print(f"num_attention_heads: {tc.num_attention_heads}  ")
print(f"num_key_value_heads: {tc.num_key_value_heads}  ")
print(f"intermediate_size:   {tc.intermediate_size}  ")
print(f"hidden_size:         {tc.hidden_size}  # locked at original")

url = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/transformers/tasks/ai2d-demo.jpg"
image = Image.open(requests.get(url, stream=True).raw).convert("RGB")
preview = image.copy(); preview.thumbnail((512, 512)); display(preview)

messages = [{"role": "user", "content": [
    {"type": "image", "image": image},
    {"type": "text", "text": "Describe the image in one short sentence."},
]}]
inputs = processor.apply_chat_template(
    messages, add_generation_prompt=True, tokenize=True, return_dict=True, return_tensors="pt"
).to(pruned.device)
out = pruned.generate(**inputs, max_new_tokens=64, do_sample=False)
print("Caption:", processor.batch_decode(out[:, inputs['input_ids'].shape[1]:], skip_special_tokens=True)[0])

import gc; del pruned; gc.collect(); torch.cuda.empty_cache()

---
## 1.3 Knowledge Distillation

Now we apply text-only knowledge distillation from the original Cosmos Reason 2 (teacher) into the pruned VLM (student), via [`distill.py`](../distill.py) from the [Megatron-Bridge](https://github.com/NVIDIA-NeMo/Megatron-Bridge) framework. The wrapper extracts both LM towers, runs `kd_loss` distillation on text tokens, then reinserts the distilled student LM back into the original VLM container.

**Cosmos Reason 2 is a reasoning VLM**, so the distillation corpus should preserve reasoning. We blend three Nemotron sources at sampling weights `0.1 / 0.5 / 0.4`:

| Dataset (HF) | Subset | Weight | Why |
|---|---|---:|---|
| `nvidia/Nemotron-Pretraining-Dataset-sample` | `Nemotron-CC-High-Quality-Synthetic` | 0.1 | General high-quality synthetic text — keeps the base language capability healthy. |
| `nvidia/Nemotron-Pretraining-SFT-v1` | `Nemotron-SFT-Math` | 0.3 | Chain-of-thought math reasoning. |
| `nvidia/Nemotron-Pretraining-SFT-v1` | `Nemotron-SFT-Code` | 0.2 | Chain-of-thought code reasoning. |

Tokenise each separately so they end up as **three distinct `.bin`/`.idx` prefixes**, then pass them to `distill.py` via `--data_paths <w> <prefix> <w> <prefix> <w> <prefix>`. All three are pretraining-style raw-text corpora (`text` key) → use `--append_eod`. See [`../dataset/MEGATRON_DATA_PREP.md`](../../dataset/MEGATRON_DATA_PREP.md) for chat-formatted alternatives.

In [ ]:
import os, json
from itertools import islice
from datasets import load_dataset

RAW_DIR       = f"{BASE_PATH}/distill_data_cosmos/raw"
TOKENIZED_DIR = f"{BASE_PATH}/distill_data_cosmos/tokenized_cosmos"
os.makedirs(RAW_DIR, exist_ok=True)

# Streaming mode pulls parquet shards on demand, so disk usage tracks MAX_SAMPLES, not
# the source size. At gbs=512, seq=4096, 5 k iters consume ~10 B tokens; with 3 shards
# averaging ~500-1000 tokens/doc the 2 M-per-shard cap below leaves enough variety
# that the schedule does not loop. Drop back to 200_000 for a smoke run.
MAX_SAMPLES = 2_000_000

# Each tuple: (hf_dataset, hf_subset, output_jsonl_stem, sampling_weight).
SOURCES = [
    ("nvidia/Nemotron-Pretraining-Dataset-sample", "Nemotron-CC-High-Quality-Synthetic", "nemotron_cc",  0.5),
    ("nvidia/Nemotron-Pretraining-SFT-v1",         "Nemotron-SFT-MATH",                  "nemotron_math", 0.3),
    ("nvidia/Nemotron-Pretraining-SFT-v1",         "Nemotron-SFT-Code",                  "nemotron_code", 0.2),
]

for hf, subset, stem, _ in SOURCES:
    jsonl_path = f"{RAW_DIR}/{stem}.jsonl"
    if os.path.exists(jsonl_path) and os.path.getsize(jsonl_path) > 0:
        print(f"  {stem:<14} (cached) → {jsonl_path}")
        continue
    print(f"  {stem:<14} streaming up to {MAX_SAMPLES:,} docs from {hf}::{subset}...")
    ds = load_dataset(hf, subset, split="train", streaming=True)
    n = 0
    with open(jsonl_path, "w") as f:
        for row in islice(ds, MAX_SAMPLES):
            text = row.get("text", "")
            if not text or not text.strip():
                continue
            f.write(json.dumps({"text": text}) + "\n")
            n += 1
            if n % 10_000 == 0:
                print(f"    ... wrote {n:,}")
    print(f"  {stem:<14} {n:>7} docs → {jsonl_path}")

In [ ]:
# Tokenise all three JSONLs in one shot. megatron_preprocess_data with multiple
# --jsonl_paths produces one .bin/.idx per input, so we get three separately-weightable
# prefixes that we'll plumb into distill.py below.
PRUNED_PATH = f"{BASE_PATH}/Cosmos-Reason2-pruned-depth-24"
TOKENIZED_DIR = f"{BASE_PATH}/distill_data_cosmos/tokenized_cosmos-2"

!python -m modelopt.torch.utils.plugins.megatron_preprocess_data \
    --jsonl_paths \
        {BASE_PATH}/distill_data_cosmos/raw/nemotron_cc.jsonl \
        {BASE_PATH}/distill_data_cosmos/raw/nemotron_math.jsonl \
        {BASE_PATH}/distill_data_cosmos/raw/nemotron_code.jsonl \
    --json_keys text \
    --tokenizer {PRUNED_PATH} \
    --output_dir {TOKENIZED_DIR} \
    --workers 8 \
    --append_eod

!ls {TOKENIZED_DIR}

Run the next cell to start distillation! 
Expect 15-30 min to complete.

Note that we are only training the model for 150 iterations. Ideally it would need to be trained longer until convergence for better results.

In [ ]:
COSMOS_PATH   = f"{BASE_PATH}/Cosmos-Reason2-2B"
PRUNED_PATH   = f"{BASE_PATH}/Cosmos-Reason2-pruned-depth-24"
TOKENIZED_DIR = f"{BASE_PATH}/distill_data_cosmos/tokenized_cosmos-2"
DISTILL_WORK_DIR = f"{BASE_PATH}/Cosmos-Reason2-pruned-depth-distill"
DISTILLED_PATH   = f"{BASE_PATH}/Cosmos-Reason2-distilled"

# Sampling weights must sum to 1.0; distill.py mixes them per training step.
# Blend is math-heavy (0.5) + code (0.4) + general CC (0.1) to favour reasoning recovery.
DATA_PATHS = (
    f"0.1 {TOKENIZED_DIR}/nemotron_cc_text "
    f"0.5 {TOKENIZED_DIR}/nemotron_math_text "
    f"0.4 {TOKENIZED_DIR}/nemotron_code_text"
)

!torchrun --nproc_per_node 4 ../distill.py \
    --tp_size 1 \
    --student_hf_path {PRUNED_PATH} \
    --teacher_hf_path {COSMOS_PATH} \
    --data_paths {DATA_PATHS} \
    --data_path_to_cache {TOKENIZED_DIR}/cache \
    --seq_length 2048 --mbs 4 --gbs 1024 \
    --train_iters 50 \
    --lr 3e-5 --min_lr 1e-5 --lr_warmup_iters 0 \
    --log_interval 1 \
    --output_dir {DISTILL_WORK_DIR} \
    --hf_export_path {DISTILLED_PATH} \
    --student_hf_model {PRUNED_PATH}

In [ ]:
# Verify that the distillation has completed and the final checkpoint is saved in the HF format.
!ls "{BASE_PATH}/Cosmos-Reason2-distilled"

Again, let's verify that the distilled VLM can generate a coherent caption.

In [ ]:
from transformers import AutoModelForImageTextToText, AutoProcessor
from IPython.display import display
from PIL import Image
import requests, torch

vlm = AutoModelForImageTextToText.from_pretrained(
    DISTILLED_PATH, dtype=torch.bfloat16, device_map="auto", trust_remote_code=True
)
processor = AutoProcessor.from_pretrained(DISTILLED_PATH, trust_remote_code=True)

url = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/transformers/tasks/ai2d-demo.jpg"
image = Image.open(requests.get(url, stream=True).raw).convert("RGB")
preview = image.copy(); preview.thumbnail((512, 512)); display(preview)

messages = [{"role": "user", "content": [
    {"type": "image", "image": image},
    {"type": "text", "text": "Describe the image in one short sentence."},
]}]
inputs = processor.apply_chat_template(
    messages, add_generation_prompt=True, tokenize=True, return_dict=True, return_tensors="pt"
).to(vlm.device)
out = vlm.generate(**inputs, max_new_tokens=64, do_sample=False)
print("Caption:", processor.batch_decode(out[:, inputs['input_ids'].shape[1]:], skip_special_tokens=True)[0])

import gc; del vlm; gc.collect(); torch.cuda.empty_cache()

In [ ]:
# Now lets launch tensorboard to inspect the loss convergence
# !tensorboard --logdir /workspace/user_homes/lmikaelyan/Model-Optimizer/examples/megatron_bridge/notebooks/Cosmos-Reason2-pruned-depth-distill

---
## 1.4 Post-training Quantization (FP8)

Quantization is a powerful technique for compressing deep learning models by reducing the precision of the numerical values used to represent model parameters, activations, and intermediate data. Instead of relying on high-precision formats like FP32 (32-bit floating point), quantization transforms these values into lower-bit representations such as FP8, INT8, NVFP4, INT4, or even down to binary (1-bit). This process unlocks several compelling advantages:
- Dramatic reduction in overall model footprint
- Lower memory bandwidth requirements
- Faster inference, especially on GPUs and resource-constrained edge devices

There are three primary strategies for quantizing models:
- **Post-Training Quantization (PTQ)**: This approach converts a fully trained model to a lower-precision format after training is complete.
- **Quantization-Aware Training (QAT)**: Here quantization is integrated directly into the training process. During both the forward and backward passes, quantization and dequantization operations are simulated, making the model robust to quantization noise that it will encounter during inference.
- **Quantization-Aware Distillation (QAD)**: Extends QAT by training a low-precision student model under the supervision of a high-precision teacher, using a distillation loss tailored to the quantized student. This helps recover near–full-precision accuracy even at very low bit-widths like FP8 and NVFP4.

In your opinion, which method do you think better preserves model accuracy?

In this section, we are going to apply post-training FP8 quantization to the distilled model we obtained earlier.

For PTQ we use [`../../llm_ptq/hf_ptq.py`](../../llm_ptq/hf_ptq.py). 

`--calib_with_images` builds the calibration set from image-text pairs so activation ranges in the projector → LM interface reflect real multimodal traffic — important for a VLM whose forward pass mixes visual and text tokens. NVFP4 is available with `--qformat nvfp4` on Blackwell.

FP8 on a reasoning VLM typically lands within 1 absolute point of BF16 on MathVista.

We are going to create FP8 checkpoints for the original model and the distilled one for a full comparison.

In [ ]:
# Original model quantization
FP8_PATH_ORIG = f"{BASE_PATH}/Cosmos-Reason2-fp8"

!python ../../llm_ptq/hf_ptq.py \
    --pyt_ckpt_path "{BASE_PATH}/Cosmos-Reason2-2B" \
    --export_path   {FP8_PATH_ORIG} \
    --qformat fp8 \
    --kv_cache_qformat none \
    --calib_with_images \
    --calib_size 512 \
    --trust_remote_code

In [ ]:
# Distilled model quantization
FP8_PATH = f"{BASE_PATH}/Cosmos-Reason2-distilled-fp8"

!python ../../llm_ptq/hf_ptq.py \
    --pyt_ckpt_path {DISTILLED_PATH} \
    --export_path   {FP8_PATH} \
    --qformat fp8 \
    --kv_cache_qformat none \
    --calib_with_images \
    --calib_size 512 \
    --trust_remote_code

In [ ]:
# Verify Quantized model exported Cosmos-Reason2-distilled-fp8
!ls "{BASE_PATH}/Cosmos-Reason2-distilled-fp8"

---
## 1.5 Serve with vLLM

Now that the compressed models are ready, we can start a vLLM OpenAI-compatible endpoint on the FP8 export. Run this in a separate terminal (or backgrounded), then return here to evaluate.

```bash
vllm serve {BASE_PATH}/Cosmos-Reason2-distilled-fp8 \
    --trust-remote-code --dtype bfloat16 \
    --mm-processor-kwargs '{"max_pixels": 2097152, "min_pixels": 262144, "max_num_frames": 1}'
```

The `--mm-processor-kwargs` cap is important for a reasoning VLM that routinely sees dense diagrams — without it a single high-resolution image can blow the per-request VRAM budget.

In [ ]:
# Poll vLLM until it's ready.
import requests, time
for _ in range(120):
    try:
        if requests.get("http://localhost:8000/health", timeout=2).status_code == 200:
            print("vLLM is ready"); break
    except requests.RequestException:
        pass
    time.sleep(5)
else:
    print("vLLM not reachable after 10 min — check the serve logs")

Once the pruned, distilled and quantized model is deployed, we are once again going to verify that it is still able to generate captions for images. How do you expect the result to change?

In [ ]:
import requests

served_model = requests.get("http://localhost:8000/v1/models").json()["data"][0]["id"]
print(f"served: {served_model}")

resp = requests.post(
    "http://localhost:8000/v1/chat/completions",
    headers={"Authorization": "Bearer token-abc123"},
    timeout=120,
    json={
        "model": served_model,
        "messages": [{
            "role": "user",
            "content": [
                {"type": "image_url", "image_url": {
                    "url": "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/transformers/tasks/ai2d-demo.jpg"
                }},
                {"type": "text", "text": "Describe the image in one short sentence."},
            ],
        }],
        "max_tokens": 64,
        "temperature": 0,
    },
).json()
print("FP8 caption:", resp["choices"][0]["message"]["content"])
print(f"tokens: prompt={resp['usage']['prompt_tokens']}, completion={resp['usage']['completion_tokens']}")

---
## 1.6 Evaluate — 4-way comparison across 3 benchmark families

In the final section, the goal is to show that the compressed checkpoint (pruned + distilled + FP8) recovers almost all of the baseline 2B BF16 quality. To make the recovery delta legible we evaluate **four checkpoints** on **three benchmark families** with a single eval driver:

| Model Variant | Path | What it tells us |
|---|---|---|
| **baseline** | `nvidia/Cosmos-Reason2-2B` (BF16) | Reference number to recover. |
| **distilled_bf16** | `{DISTILLED_PATH}` (BF16) | What KD recovers before quantization. |
| **distilled_fp8** | `{FP8_PATH}` | Shippable artifact. |
| **orig_fp8** | `{FP8_PATH_ORIG}` | Original model quantized. |

| Benchmark | Why | Task names (lmms-eval) |
|---|---|---|
| **MathVista (testmini)** | Reasoning headline — Cosmos Reason 2's main specialisation. | `mathvista_testmini` |
| **PAI-Bench (BLINK proxies)** | Cosmos's intended Physical-AI domain. BLINK-Depth + BLINK-Spatial are the publicly reproducible parts of the PAI-Bench leaderboard. | `blink` (subtask filters in cell below) |
| **RealWorldQA** | General-purpose cross-check — guards against breaking non-reasoning behavior. | `realworldqa` |

We loop **vLLM-up → run all 3 benchmarks → vLLM-down** per variant. The §1.5 manual `vllm serve` command is wrapped in a helper that the driver calls. Expect ~30-40 min total per variant (≈ 2-3 h end-to-end on 1 H100).

> Stop the `vllm serve` you started in §1.5 before running this section — the driver manages its own vLLM lifecycle on the same port.

Now we are ready to run evaluation! The following cells serves each model variants and evaluates them on each benchmark.

Allow 15-20 min for this cell to complete.

In [ ]:
# 4-way × 3-benchmark driver. Brings vLLM up per variant, runs lmms-eval against all
# three benchmark families, tears vLLM down. Results land under EVAL_ROOT/<variant>/.
import os, subprocess, time, shlex, json, signal
import requests

EVAL_ROOT = f"{BASE_PATH}/logs/cosmos-r2-recovery-eval"
os.makedirs(EVAL_ROOT, exist_ok=True)

VARIANTS = {
    "baseline":        COSMOS_PATH,          # unpruned Cosmos-Reason2-2B (BF16)
    "distilled_bf16":  DISTILLED_PATH,       # pruned + distilled, BF16
    "distilled_fp8":   FP8_PATH,             # pruned + distilled + FP8 (final)
    "orig_fp8": FP8_PATH_ORIG,
}
# BENCHMARKS = {
#     "mathvista":   ["mathvista_testmini"],
#     "realworldqa": ["realworldqa"],
#     "paibench":    ["blink"],           # set in the discovery cell above
BENCHMARKS = {
    "realworldqa": ["realworldqa"],
    "paibench":    ["blink"],           # set in the discovery cell above
}
MM_KW = '{"max_pixels": 2097152, "min_pixels": 262144, "max_num_frames": 1}'


def serve_vllm(ckpt: str):
    """Start vLLM in the background, return the Popen handle."""
    cmd = (
        f"vllm serve {shlex.quote(ckpt)} "
        "--trust-remote-code --dtype bfloat16 "
        f"--mm-processor-kwargs {shlex.quote(MM_KW)}"
    )
    return subprocess.Popen(cmd, shell=True, preexec_fn=os.setsid)


def wait_ready(timeout_s: int = 600):
    for _ in range(timeout_s // 5):
        try:
            if requests.get("http://localhost:8000/v1/models", timeout=2).ok:
                return True
        except requests.RequestException:
            pass
        time.sleep(5)
    return False


def stop_vllm(p):
    try:
        os.killpg(os.getpgid(p.pid), signal.SIGTERM)
    except ProcessLookupError:
        pass
    p.wait(timeout=60)


def run_lmms_eval(ckpt: str, tasks: list[str], out_dir: str):
    if not tasks:
        return
    os.makedirs(out_dir, exist_ok=True)
    env = {**os.environ, "OPENAI_API_KEY": "token-abc123"}
    cmd = [
        "python", "-m", "lmms_eval",
        "--model", "openai",
        "--model_args", f"model={ckpt},base_url=http://localhost:8000/v1",
        "--tasks", ",".join(tasks),
        "--batch_size", "32",
        "--output_path", out_dir,
    ]
    subprocess.run(cmd, env=env, check=False)


for variant, ckpt in VARIANTS.items():
    print(f"\n{'='*60}\n>>> {variant}  ({ckpt})\n{'='*60}")
    proc = serve_vllm(ckpt)
    try:
        if not wait_ready():
            print(f"!! vLLM failed to come up for {variant}; skipping.")
            continue
        for bench_name, tasks in BENCHMARKS.items():
            out_dir = f"{EVAL_ROOT}/{variant}/{bench_name}"
            print(f"  -> {bench_name}: {tasks}")
            run_lmms_eval(ckpt, tasks, out_dir)
    finally:
        stop_vllm(proc)

print(f"\nAll variants done. Results under {EVAL_ROOT}/")

### Aggregate the comparison

Collect per-variant scores and render the recovery table. `% recovered` is

$$
\\text{{recovered}} = \\frac{{\\text{{variant}} - \\text{{pruned\\_only}}}}{{\\text{{baseline}} - \\text{{pruned\\_only}}}}
$$

so the pruned-only baseline anchors 0 % and the BF16 baseline anchors 100 %. The shippable artifact is `distilled_fp8`; we want its row to be ≥ 90 % recovered on every benchmark and within 2 absolute points of baseline on MathVista.

In [ ]:
# Walk EVAL_ROOT/<variant>/<bench>/ for lmms-eval result JSONs and render a table.
import os, json, glob

EVAL_ROOT = f"{BASE_PATH}/logs/cosmos-r2-recovery-eval"
VARIANT_ORDER = ["baseline", "pruned_only", "distilled_bf16", "distilled_fp8"]
BENCH_KEYS = ["mathvista", "paibench", "realworldqa"]


def headline_metric(results_json: dict, task_substring: str) -> float | None:
    """Pull the first 'exact_match,*' or 'overall,*' metric for a task matching substring."""
    if "results" not in results_json:
        return None
    for task_name, metrics in results_json["results"].items():
        if task_substring not in task_name:
            continue
        for k, v in metrics.items():
            if (k.startswith("exact_match") or k.startswith("overall") or k.startswith("acc")) and not k.endswith("stderr") and isinstance(v, (int, float)):
                return float(v) * (100 if v <= 1 else 1)
    return None


def latest_result(variant: str, bench: str) -> dict | None:
    candidates = sorted(glob.glob(f"{EVAL_ROOT}/{variant}/{bench}/**/results*.json", recursive=True))
    if not candidates:
        return None
    with open(candidates[-1]) as f:
        return json.load(f)


scores: dict[str, dict[str, float | None]] = {v: {} for v in VARIANT_ORDER}
TASK_MATCH = {"mathvista": "mathvista", "paibench": "blink", "realworldqa": "realworldqa"}
for v in VARIANT_ORDER:
    for b in BENCH_KEYS:
        rj = latest_result(v, b)
        scores[v][b] = headline_metric(rj, TASK_MATCH[b]) if rj else None


def pct(num: float | None) -> str:
    return f"{num:5.1f}" if isinstance(num, float) else "  ?  "


print(f"{'variant':<18} | " + " | ".join(f"{b:>11}" for b in BENCH_KEYS))
print("-" * 70)
for v in VARIANT_ORDER:
    row = " | ".join(pct(scores[v][b]) for b in BENCH_KEYS)
    print(f"{v:<18} | {row}")

print("\n% recovered (vs. baseline, anchored at pruned_only=0%):")
print(f"{'variant':<18} | " + " | ".join(f"{b:>11}" for b in BENCH_KEYS))
print("-" * 70)
for v in VARIANT_ORDER:
    cells = []
    for b in BENCH_KEYS:
        base = scores["baseline"].get(b)
        prune = scores["pruned_only"].get(b)
        cur = scores[v].get(b)
        if None in (base, prune, cur) or base == prune:
            cells.append("   -   ")
            continue
        cells.append(f"{(cur - prune) / (base - prune) * 100:>9.1f}%")
    print(f"{v:<18} | " + " | ".join(cells))

---
## 1.7 Inference perf benchmark — aiperf across the deployable variants

Accuracy is half the story. Compression is worthwhile because the smaller / lower-precision model **serves faster**. We run [aiperf](https://github.com/ai-dynamo/aiperf) against a live vLLM endpoint with a **realistic VLM workload — image + text prompt** — and measure:

| Metric | What it tells you |
|---|---|
| **TTFT** (time-to-first-token) | Prefill latency — for a VLM this is dominated by the vision tower + image-token prefill |
| **ITL** (inter-token latency)  | Decode latency — per-token throughput per request |
| **Request throughput**          | End-to-end requests/sec at a given concurrency |
| **Output-token throughput**     | Aggregate tokens/sec across all concurrent requests |


**Workload:** aiperf builds chat requests with `--image-batch-size 1` (one synthetic image per request, resized from `assets/source_images`) at **768 × 768 px** — within the [`min_pixels=262144`, `max_pixels=2097152`] envelope you set on the vLLM server in §5.5, so each request actually exercises the vision tower. Text portion is a short instruction (~80 tokens), output capped at 100 tokens — a representative "describe this image" turn.

> 📦 If aiperf isn't installed: `pip install --ignore-installed aiperf`. Default sweep is concurrency 16 + 64 (~10-15 min per variant); bump `CONCURRENCIES` to `[4, 8, 16, 32, 64, 128]` for the full notebook-4 curve.

> 🔥 Stop the manual `vllm serve` from §5.5 first — the driver below manages its own vLLM lifecycle on port 8000.

In [ ]:
# §5.7.a aiperf sweep driver — one vLLM lifecycle per variant, image + text workload.
import os, subprocess, time, shlex, signal
import requests

BENCH_ROOT = f"{BASE_PATH}/benchmarks/cosmos-r2-aiperf"
os.makedirs(BENCH_ROOT, exist_ok=True)

# Skip pruned_only (not a deployment target). Order matters: baseline first so the
# unpruned model's KV-cache + cuDNN selections warm the disk page cache for the
# subsequent runs.
VARIANTS = {
    "baseline":       COSMOS_PATH,        # unpruned BF16
    "orig_fp8":       FP8_PATH_ORIG,      # FP8 only, no distill
    "distilled_bf16": DISTILLED_PATH,     # pruned + distilled, BF16
    "distilled_fp8":  FP8_PATH,           # pruned + distilled + FP8 (shippable)
}
CONCURRENCIES = [16, 64]   # bump to [4, 8, 16, 32, 64, 128] for the full curve
IMG_W, IMG_H  = 768, 768   # ~590 K pixels, inside [min_pixels, max_pixels]
TEXT_ISL      = 80         # text tokens around the image
OSL           = 100        # output tokens

MM_KW = '{"max_pixels": 2097152, "min_pixels": 262144, "max_num_frames": 1}'


def serve_vllm(ckpt: str, log_path: str):
    """Start vLLM in its own process group, log to file."""
    cmd = (
        f"vllm serve {shlex.quote(ckpt)} "
        "--trust-remote-code --dtype bfloat16 "
        f"--mm-processor-kwargs {shlex.quote(MM_KW)}"
    )
    f = open(log_path, "w")
    return subprocess.Popen(cmd, shell=True, stdout=f, stderr=subprocess.STDOUT, preexec_fn=os.setsid)


def wait_ready(timeout_s: int = 600) -> bool:
    for _ in range(timeout_s // 5):
        try:
            if requests.get("http://localhost:8000/v1/models", timeout=2).ok:
                return True
        except requests.RequestException:
            pass
        time.sleep(5)
    return False


def stop_vllm(p):
    try:
        os.killpg(os.getpgid(p.pid), signal.SIGTERM)
        p.wait(timeout=60)
    except (ProcessLookupError, subprocess.TimeoutExpired):
        try: os.killpg(os.getpgid(p.pid), signal.SIGKILL)
        except ProcessLookupError: pass


def run_aiperf(ckpt: str, concurrency: int, out_dir: str):
    """One aiperf profile run at a single concurrency level."""
    os.makedirs(out_dir, exist_ok=True)
    cmd = [
        "aiperf", "profile",
        "--model", ckpt,
        "--url", "http://localhost:8000",
        "--endpoint-type", "chat",
        "--streaming",
        "--concurrency", str(concurrency),
        # Send 8× concurrency requests so steady-state metrics are stable.
        "--request-count", str(concurrency * 8),
        "--warmup-request-count", str(concurrency),
        # Image + text workload — this is the bit that actually exercises the VLM.
        "--image-batch-size", "1",
        "--image-width-mean",  str(IMG_W),
        "--image-height-mean", str(IMG_H),
        "--image-width-stddev", "0",
        "--image-height-stddev", "0",
        "--synthetic-input-tokens-mean", str(TEXT_ISL),
        "--output-tokens-mean", str(OSL),
        "--tokenizer", ckpt,
        "--tokenizer-trust-remote-code",
        "--output-artifact-dir", out_dir,
    ]
    log = open(f"{out_dir}/aiperf.log", "w")
    subprocess.run(cmd, stdout=log, stderr=subprocess.STDOUT, check=False)


for variant, ckpt in VARIANTS.items():
    print(f"\n{'='*60}\n>>> {variant}  ({ckpt})\n{'='*60}")
    vllm_log = f"{BENCH_ROOT}/{variant}_vllm.log"
    proc = serve_vllm(ckpt, vllm_log)
    print(f"  vllm pid={proc.pid}  log={vllm_log}")
    try:
        if not wait_ready():
            print(f"  !! vLLM did not come up; skipping {variant}.")
            continue
        for c in CONCURRENCIES:
            out_dir = f"{BENCH_ROOT}/{variant}/c{c}"
            print(f"  -> aiperf concurrency={c}  out={out_dir}")
            run_aiperf(ckpt, c, out_dir)
    finally:
        stop_vllm(proc)
        time.sleep(5)   # give NCCL / shm cleanup a moment before next variant

print(f"\nAll variants done. Artifacts under {BENCH_ROOT}/")

In [ ]:
# §5.7.b Aggregate aiperf results — TTFT / ITL / RPS / TPS per (variant, concurrency).
import os, json, glob
from pathlib import Path

BENCH_ROOT = f"{BASE_PATH}/benchmarks/cosmos-r2-aiperf"
VARIANT_ORDER = ["baseline", "orig_fp8", "distilled_bf16", "distilled_fp8"]


def find_profile_json(out_dir: str) -> str | None:
    # aiperf writes profile_export_aiperf.json into the artifact dir (sometimes nested).
    matches = sorted(glob.glob(f"{out_dir}/**/profile_export_aiperf.json", recursive=True))
    return matches[-1] if matches else None


def pull_metric(d: dict, *path):
    cur = d
    for k in path:
        if not isinstance(cur, dict) or k not in cur:
            return None
        cur = cur[k]
    return cur


rows = []
for variant in VARIANT_ORDER:
    var_dir = f"{BENCH_ROOT}/{variant}"
    if not os.path.isdir(var_dir):
        continue
    for c_dir in sorted(Path(var_dir).iterdir()):
        if not c_dir.name.startswith("c"):
            continue
        c = int(c_dir.name[1:])
        f = find_profile_json(str(c_dir))
        if not f:
            rows.append({"variant": variant, "concurrency": c, "ttft": None, "itl": None, "rps": None, "tps": None})
            continue
        d = json.load(open(f))
        # aiperf reports metrics under top-level keys with sub-fields {avg, p50, p90, p95, p99, ...}
        rows.append({
            "variant":     variant,
            "concurrency": c,
            "ttft":        pull_metric(d, "time_to_first_token",     "avg"),
            "itl":         pull_metric(d, "inter_token_latency",     "avg"),
            "rps":         pull_metric(d, "request_throughput",      "avg"),
            "tps":         pull_metric(d, "output_token_throughput", "avg"),
        })


def fmt(x, unit=""):
    return f"{x:8.2f}{unit}" if isinstance(x, (int, float)) else f"{'  ?  ':>8}{unit}"


print(f"{'variant':<16} | {'conc':>5} | {'TTFT (ms)':>11} | {'ITL (ms)':>11} | {'RPS':>10} | {'TPS (tok/s)':>12}")
print("-" * 80)
for r in rows:
    print(f"{r['variant']:<16} | {r['concurrency']:>5} | {fmt(r['ttft'])}    | {fmt(r['itl'])}    | {fmt(r['rps'])} | {fmt(r['tps'])}")

# Speedup vs baseline at matching concurrency, on the output-token-throughput axis
# (the headline number for "how much faster does this serve").
print("\nSpeedup vs baseline (tok/s ratio):")
baseline_tps = {r["concurrency"]: r["tps"] for r in rows if r["variant"] == "baseline"}
for r in rows:
    if r["variant"] == "baseline" or r["tps"] is None or baseline_tps.get(r["concurrency"]) is None:
        continue
    print(f"  {r['variant']:<16} @ c={r['concurrency']:>3}:  {r['tps'] / baseline_tps[r['concurrency']]:5.2f}×")

---
## 5.7 Shutdown

Stop the vLLM server you started in §5.5.

In [ ]:
!pkill -f 'vllm serve' || true
!nvidia-smi --query-gpu=memory.used --format=csv,noheader

---
## Wrap-up

What just happened:

1. **Pruned** Cosmos Reason 2 on the LM tower from 1.7 B → ~1.4 B (15 % cut on the LM; full VLM 2.0 B → ~1.7 B since vision/projector is untouched) by NAS-searching over layer count, attention heads, KV groups and FFN width (hidden_size locked).
2. **Distilled** the pruned student against the unpruned teacher on a Nemotron CC + Math + Code blend — 150 iters on 4×H100 — enough for a smoke-test run; bump to 5 000+ iters for full quality recovery.
3. **Quantized** the distilled VLM to FP8 with image-aware calibration; the resulting checkpoint serves directly with vLLM at ~2× the BF16 throughput on H100.
4. **Evaluated** all four variants (baseline / pruned-only / distilled / distilled+FP8) on MathVista, BLINK (PAI-Bench proxy), and RealWorldQA, and aggregated `% recovered` into a single table.

The full path uses the same scripts and helpers as notebooks 01-04 — the only differences for Cosmos Reason 2 are the model id, the use of `--prune_export_config` for depth pruning, the Nemotron reasoning corpus over wikitext, and the 4-way comparison against three reasoning-aware benchmarks. Everything else (the VLM extract/reinsert wrappers, the `hidden_size` invariant guard, the FP8 calibration flow, the vLLM serve recipe) is reused without modification.

**Levers if MathVista falls short of the recovery target**, in order of expected impact:

1. Bump `--train_iters` 150 → 5000+ for full convergence (the 150-iter run here is a smoke test).
2. Raise `--gbs` 512 → 1024 if memory allows (halves effective epochs per iter; helps when the corpus is large).
3. Tilt the corpus blend toward math: 0.5/0.3/0.2 → 0.4/0.4/0.2.
4. Add a short FP8 QAT pass after §4 (out of scope for this notebook; see `examples/llm_qat/`).